# Reward Model: Fine-tune Encoder (Bradley-Terry loss)

**Запускать на Google Colab с T4 GPU**

Идея: вместо frozen embeddings → MLP обучаем сам трансформер напрямую.  
Модель получает пару (chosen, rejected) и учится давать chosen более высокий скор через Bradley-Terry loss:
```
L = -log( σ(r_chosen - r_rejected) )
```

## 0. GPU check + install

In [ ]:
import subprocess, sys

# Проверка GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND — смени Runtime на T4!')

# Install
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'datasets', 'transformers', 'accelerate'], check=True)
print('Packages ready.')

## 1. Config — все гиперпараметры в одном месте

In [ ]:
CFG = dict(
    # Модель
    model_name   = 'roberta-base',
    max_length   = 512,

    # Данные
    train_size   = 40_000,           # None = полный датасет (~7ч на T4, рискованно)
    val_size     = 4_000,
    seed         = 42,

    # Обучение
    epochs             = 10,         # early stopping остановит раньше
    batch_size         = 128,        # train батч
    val_batch_size     = 64,         # val батч отдельно — не зависит от train
    grad_accum_steps   = 1,          # effective batch = 64 (теперь за один шаг)
    lr                 = 2e-5,
    weight_decay       = 0.01,
    warmup_ratio       = 0.06,
    max_grad_norm      = 1.0,
    fp16               = True,

    # Сохранение
    output_dir   = '/content/reward_model_bt',
    save_steps   = 500,
)

print('Config set.')
for k, v in CFG.items():
    print(f'  {k:22s} = {v}')


## 2. Imports

In [ ]:
import os, math, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from datasets import load_dataset
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3. Данные

In [ ]:
raw = load_dataset('Anthropic/hh-rlhf')
print(raw)

rng = np.random.default_rng(CFG['seed'])

train_data = raw['train']
if CFG['train_size'] is not None:
    idx = rng.choice(len(train_data), CFG['train_size'] + CFG['val_size'], replace=False)
    train_idx = idx[:CFG['train_size']].tolist()
    val_idx   = idx[CFG['train_size']:].tolist()
    train_split = train_data.select(train_idx)
    val_split   = train_data.select(val_idx)
else:
    # Используем официальный test как val
    train_split = train_data
    val_split   = raw['test']

print(f'Train: {len(train_split)} | Val: {len(val_split)}')

## 4. Tokenizer + Dataset

In [ ]:
# truncation_side='left' — режем начало, сохраняем конец
# В hh-rlhf разница между chosen/rejected в последнем ответе ассистента (в хвосте текста)
tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
tokenizer.truncation_side = 'left'


class PairwiseDataset(Dataset):
    """Возвращает токенизированные пары (chosen, rejected)."""

    def __init__(self, hf_dataset, tokenizer, max_length):
        self.data      = hf_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def _tokenize(self, text):
        return self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )

    def __getitem__(self, idx):
        row = self.data[idx]
        c = self._tokenize(row['chosen'])
        r = self._tokenize(row['rejected'])
        return {
            'chosen_input_ids':      c['input_ids'].squeeze(0),
            'chosen_attention_mask': c['attention_mask'].squeeze(0),
            'rejected_input_ids':      r['input_ids'].squeeze(0),
            'rejected_attention_mask': r['attention_mask'].squeeze(0),
        }


train_ds = PairwiseDataset(train_split, tokenizer, CFG['max_length'])
val_ds   = PairwiseDataset(val_split,   tokenizer, CFG['max_length'])

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['val_batch_size'], shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')


## 5. Reward Model

In [ ]:
FREEZE_LAYERS = 8   # заморозить нижние N слоёв из 12 (roberta-base)
                     # обучаем только верхние 4 слоя + голову → меньше оверфит

class RewardModel(nn.Module):
    def __init__(self, model_name, freeze_layers=FREEZE_LAYERS):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

        # Замораживаем embeddings и нижние freeze_layers трансформер-блоков
        for param in self.encoder.embeddings.parameters():
            param.requires_grad = False
        for i, layer in enumerate(self.encoder.encoder.layer):
            if i < freeze_layers:
                for param in layer.parameters():
                    param.requires_grad = False

        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.reward_head = nn.Linear(hidden, 1)
        nn.init.normal_(self.reward_head.weight, std=0.02)
        nn.init.zeros_(self.reward_head.bias)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]      # (B, H)
        reward = self.reward_head(self.dropout(cls)).squeeze(-1)  # (B,)
        return reward


model = RewardModel(CFG['model_name']).to(device)
total_params    = sum(p.numel() for p in model.parameters())
trainable       = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen          = total_params - trainable
print(f'Total:     {total_params/1e6:.1f}M')
print(f'Trainable: {trainable/1e6:.1f}M  (обучаем)')
print(f'Frozen:    {frozen/1e6:.1f}M  (заморожено)')


## 6. Bradley-Terry Loss + Optimizer + Scheduler

In [ ]:
def bradley_terry_loss(r_chosen: torch.Tensor, r_rejected: torch.Tensor) -> torch.Tensor:
    """
    L = -log( sigmoid(r_chosen - r_rejected) )

    Модель учится: r_chosen > r_rejected.
    Чем больше разрыв в правильную сторону — тем ниже loss.
    """
    return -torch.nn.functional.logsigmoid(r_chosen - r_rejected).mean()


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG['lr'],
    weight_decay=CFG['weight_decay'],
    eps=1e-6,
)

total_steps   = math.ceil(len(train_loader) / CFG['grad_accum_steps']) * CFG['epochs']
warmup_steps  = int(total_steps * CFG['warmup_ratio'])

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

scaler = GradScaler(enabled=CFG['fp16'])

print(f'Total optimizer steps: {total_steps} | Warmup: {warmup_steps}')

## 7. Training Loop

In [ ]:
os.makedirs(CFG['output_dir'], exist_ok=True)


def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch in loader:
            c_ids  = batch['chosen_input_ids'].to(device)
            c_mask = batch['chosen_attention_mask'].to(device)
            r_ids  = batch['rejected_input_ids'].to(device)
            r_mask = batch['rejected_attention_mask'].to(device)

            with autocast(enabled=CFG['fp16']):
                r_c = model(c_ids, c_mask)
                r_r = model(r_ids, r_mask)
                loss = bradley_terry_loss(r_c, r_r)

            total_loss += loss.item() * len(c_ids)
            correct    += (r_c > r_r).sum().item()
            total      += len(c_ids)

    return total_loss / total, correct / total


best_val_acc  = 0.0
best_val_loss = float('inf')
patience      = 2          # эпох без улучшения val_loss до остановки
patience_ctr  = 0
global_step   = 0
history       = []

for epoch in range(1, CFG['epochs'] + 1):
    model.train()
    epoch_loss, epoch_correct, epoch_total = 0.0, 0, 0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{CFG["epochs"]}')
    for step, batch in enumerate(pbar, 1):
        c_ids  = batch['chosen_input_ids'].to(device)
        c_mask = batch['chosen_attention_mask'].to(device)
        r_ids  = batch['rejected_input_ids'].to(device)
        r_mask = batch['rejected_attention_mask'].to(device)

        with autocast(enabled=CFG['fp16']):
            r_c = model(c_ids, c_mask)
            r_r = model(r_ids, r_mask)
            loss = bradley_terry_loss(r_c, r_r) / CFG['grad_accum_steps']

        scaler.scale(loss).backward()

        epoch_loss    += loss.item() * CFG['grad_accum_steps'] * len(c_ids)
        epoch_correct += (r_c.detach() > r_r.detach()).sum().item()
        epoch_total   += len(c_ids)

        if step % CFG['grad_accum_steps'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['max_grad_norm'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            pbar.set_postfix({
                'loss': f'{epoch_loss / epoch_total:.4f}',
                'acc':  f'{epoch_correct / epoch_total:.4f}',
                'lr':   f'{scheduler.get_last_lr()[0]:.2e}',
            })

            # Промежуточное сохранение
            if global_step % CFG['save_steps'] == 0:
                ckpt = os.path.join(CFG['output_dir'], f'ckpt_step{global_step}')
                model.encoder.save_pretrained(ckpt)
                tokenizer.save_pretrained(ckpt)
                torch.save(model.reward_head.state_dict(),
                           os.path.join(ckpt, 'reward_head.pt'))
                print(f'\nCheckpoint saved: {ckpt}')

    # Валидация после каждой эпохи
    val_loss, val_acc = evaluate(model, val_loader)
    train_loss = epoch_loss / epoch_total
    train_acc  = epoch_correct / epoch_total

    print(f'\nEpoch {epoch} | '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

    history.append(dict(epoch=epoch, train_loss=train_loss, train_acc=train_acc,
                        val_loss=val_loss, val_acc=val_acc))

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc  = val_acc
        patience_ctr  = 0
        best_path = os.path.join(CFG['output_dir'], 'best_model')
        model.encoder.save_pretrained(best_path)
        tokenizer.save_pretrained(best_path)
        torch.save(model.reward_head.state_dict(),
                   os.path.join(best_path, 'reward_head.pt'))
        print(f'Best model saved → val_loss={val_loss:.4f} val_acc={val_acc:.4f}')
    else:
        patience_ctr += 1
        print(f'No improvement ({patience_ctr}/{patience})')
        if patience_ctr >= patience:
            print(f'Early stopping at epoch {epoch}.')
            break

    model.train()

print(f'\nDone. Best Val Loss: {best_val_loss:.4f} | Best Val Acc: {best_val_acc:.4f}')

## 8. Кривые обучения

In [ ]:
import matplotlib.pyplot as plt

epochs_range = [h['epoch'] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_range, [h['train_loss'] for h in history], label='train')
ax1.plot(epochs_range, [h['val_loss']   for h in history], label='val')
ax1.set_title('Bradley-Terry Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs_range, [h['train_acc'] for h in history], label='train')
ax2.plot(epochs_range, [h['val_acc']   for h in history], label='val')
ax2.axhline(0.5, color='gray', linestyle='--', label='random baseline')
ax2.set_title('Pairwise Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print('\nHistory:')
for h in history:
    print(f"  Epoch {h['epoch']}: train_acc={h['train_acc']:.4f} | val_acc={h['val_acc']:.4f}")

## 9. Сравнение с baseline (LogReg)

| Модель | Embedding | Данные | Val Acc |
|--------|-----------|--------|---------|
| LogReg (baseline) | MiniLM-L6 frozen | 15k | 0.627 |
| LogReg (final) | MPNet frozen | 160k | — |
| **RoBERTa fine-tuned** | **end-to-end** | **40k** | **← здесь** |

## 10. Скачать модель из Colab

In [ ]:
# Вариант 1 — zip и скачать
import shutil
shutil.make_archive('/content/reward_model_bt_best', 'zip',
                    os.path.join(CFG['output_dir'], 'best_model'))

from google.colab import files
files.download('/content/reward_model_bt_best.zip')

In [ ]:
# Вариант 2 — залить на Google Drive
from google.colab import drive
drive.mount('/content/drive')

dest = '/content/drive/MyDrive/reward_model_bt'
shutil.copytree(os.path.join(CFG['output_dir'], 'best_model'), dest, dirs_exist_ok=True)
print(f'Saved to Drive: {dest}')

## 11. Инференс — как использовать обученную модель

In [ ]:
def load_reward_model(model_dir, device='cuda'):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    rm = RewardModel.__new__(RewardModel)
    nn.Module.__init__(rm)
    rm.encoder = AutoModel.from_pretrained(model_dir)
    hidden = rm.encoder.config.hidden_size
    rm.dropout = nn.Dropout(0.1)
    rm.reward_head = nn.Linear(hidden, 1)
    rm.reward_head.load_state_dict(
        torch.load(os.path.join(model_dir, 'reward_head.pt'), map_location=device)
    )
    rm = rm.to(device).eval()
    return rm, tokenizer


def score_text(model, tokenizer, text, device='cuda', max_length=512):
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       max_length=max_length, padding=True).to(device)
    with torch.no_grad():
        reward = model(inputs['input_ids'], inputs['attention_mask'])
    return reward.item()


# Пример использования
rm, tok = load_reward_model(os.path.join(CFG['output_dir'], 'best_model'), device=device)

chosen_example  = raw['test'][0]['chosen']
rejected_example = raw['test'][0]['rejected']

r_chosen   = score_text(rm, tok, chosen_example,   device=device)
r_rejected = score_text(rm, tok, rejected_example, device=device)

print(f'Reward chosen:   {r_chosen:.4f}')
print(f'Reward rejected: {r_rejected:.4f}')
print(f'Correct ranking: {r_chosen > r_rejected}')

## 12. Финальный тест на официальной тестовой выборке

In [ ]:
print("Evaluating best model on official hh-rlhf test split (8552 pairs)...")

rm_test, tok_test = load_reward_model(
    os.path.join(CFG['output_dir'], 'best_model'), device=device
)

test_ds     = PairwiseDataset(raw['test'], tokenizer, CFG['max_length'])
test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'] * 2,
                         shuffle=False, num_workers=2, pin_memory=True)

test_loss, test_acc = evaluate(rm_test, test_loader)

print()
print("=" * 55)
print("  FINAL TEST RESULTS (official hh-rlhf test set)")
print("=" * 55)
print(f"  Test Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  Test BT Loss  : {test_loss:.4f}")
print("=" * 55)

results = [
    ("LogReg (MiniLM frozen, 15k)",  0.6273),
    ("LogReg (MPNet frozen, 160k)",  None),
    ("RoBERTa fine-tuned (40k)",     test_acc),
]
print(f"\n  {'Модель':<35} {'Test Acc':>10}")
print("-" * 55)
for name, acc in results:
    acc_str = f"{acc:.4f}" if acc is not None else "см. final_model_pipeline"
    marker  = "  <-- YOU ARE HERE" if acc == test_acc else ""
    print(f"  {name:<35} {acc_str:>10}{marker}")
